In [1]:
from claude_agent_sdk import query, ClaudeAgentOptions


async def main() -> None:
    async for msg in query(
        prompt="Reply what is teh sum of 2 and 40. And what is that number according to the literature",
        options=ClaudeAgentOptions(
            model='claude-sonnet-5-5',
            max_turns=1,
            allowed_tools=[]
        )
    ):
        if type(msg).__name__ == 'ResultMessage':
            print('result::', msg.result)

# Jupyter already runs an event loop, so await directly instead of anyio.run(main)
await main()

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


result:: 2 + 40 = **42**.

In the literature, 42 is best known as the "Answer to the Ultimate Question of Life, the Universe, and Everything" in Douglas Adams's *The Hitchhiker's Guide to the Galaxy*. The supercomputer Deep Thought works on the question for 7.5 million years and arrives at 42. Nobody knows what the actual question is.


In [ ]:
# --- A tiny RAG system: TF-IDF retrieval (pure Python) + Claude for generation ---
import math
import re
from collections import Counter

from claude_agent_sdk import query, ClaudeAgentOptions

# 1. Knowledge base: swap these for your own documents
DOCUMENTS = [
    "The Eiffel Tower is in Paris, France. It was completed in 1889 and is about 330 metres tall.",
    "Mount Fuji is the highest mountain in Japan at 3,776 metres. It is an active volcano that last erupted in 1707.",
    "Python was created by Guido van Rossum and first released in 1991. Its name comes from Monty Python.",
    "In The Hitchhiker's Guide to the Galaxy, the number 42 is the Answer to the Ultimate Question of Life, the Universe, and Everything.",
    "Tokyo is the capital of Japan and one of the most populous metropolitan areas in the world.",
    "The Great Wall of China is more than 21,000 kilometres long, counting all of its branches.",
]


# 2. Indexing: tokenize and compute TF-IDF vectors
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


class TfidfIndex:
    def __init__(self, docs: list[str]):
        self.docs = docs
        tokenized = [tokenize(d) for d in docs]
        n = len(docs)
        df = Counter(term for toks in tokenized for term in set(toks))
        self.idf = {t: math.log((1 + n) / (1 + c)) + 1 for t, c in df.items()}
        self.vectors = [self._vectorize(toks) for toks in tokenized]

    def _vectorize(self, tokens: list[str]) -> dict[str, float]:
        tf = Counter(tokens)
        vec = {t: c * self.idf.get(t, 0.0) for t, c in tf.items()}
        norm = math.sqrt(sum(v * v for v in vec.values())) or 1.0
        return {t: v / norm for t, v in vec.items()}

    def search(self, question: str, k: int = 3) -> list[tuple[float, str]]:
        q = self._vectorize(tokenize(question))
        scored = [
            (sum(w * vec.get(t, 0.0) for t, w in q.items()), doc)
            for vec, doc in zip(self.vectors, self.docs)
        ]
        return sorted((s for s in scored if s[0] > 0), reverse=True)[:k]


index = TfidfIndex(DOCUMENTS)


# 3. Retrieval + generation
async def ask(question: str, k: int = 3) -> str | None:
    hits = index.search(question, k)
    context = "\n".join(f"[{i + 1}] {doc}" for i, (_, doc) in enumerate(hits)) or "(no relevant documents)"

    print("Retrieved:")
    for score, doc in hits:
        print(f"  {score:.2f}  {doc[:80]}")

    prompt = f"Context:\n{context}\n\nQuestion: {question}"
    options = ClaudeAgentOptions(
        model="claude-sonnet-5-5",
        max_turns=1,
        allowed_tools=[],
        system_prompt=(
            "Answer the question using only the provided context. "
            "Cite sources like [1]. If the context does not contain the answer, say you don't know."
        ),
    )

    answer = None
    async for msg in query(prompt=prompt, options=options):
        if type(msg).__name__ == "ResultMessage":
            answer = msg.result
    return answer


print("\nAnswer:", await ask("How tall is Mount Fuji and when did it last erupt?"))
print("\nAnswer:", await ask("What does the number 42 mean in literature?"))